# **Data Loading:**

This notebook consists of the following steps:

1. Master setup cell (imports, load data, convert columns, create derived columns)

2. The cell that checks shape, label counts, missing values

3. The cell that creates Y_group and N_group

In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

print("All liberaries are loaded successfully")


All liberaries are loaded successfully


In [30]:
df = pd.read_excel("/content/ML targets (7).xlsx")
print('Shape of the dataset:', df.shape)
print('\nLabel counts:')
print(df['RMSD_Mimic_Target (Y)'].value_counts())
print('\nColumn Name:')
print(df.columns.tolist())
print('\nMissing values:')
print(df.isnull().sum()[df.isnull().sum()>0])


Shape of the dataset: (399, 23)

Label counts:
RMSD_Mimic_Target (Y)
Y    262
N    137
Name: count, dtype: int64

Column Name:
['Organism', 'Protein', 'Position', 'HLA Haplotype', 'Pathogen Peptide', 'pathogen length', '%Rank_EL(X)', 'Aff(nM)(X)', 'Immunogenicity', 'Type of MHC', 'Human_match', 'BLOSUM80 score', 'Identity percentage', 'Alignment length (Sequence)', 'Identical aa', 'Positions', 'Human Peptide', 'Human length', 'Alignment Length (Structure)', 'Structural RMSD', 'TM-align score (Human chain 2)', 'Structural alignment coverage %', 'RMSD_Mimic_Target (Y)']

Missing values:
Series([], dtype: int64)


In [31]:
df['BLOSUM80 score'] = pd.to_numeric(df['BLOSUM80 score'], errors='coerce')
df['pathogen length'] = pd.to_numeric(df['pathogen length'], errors='coerce')
df['Alignment length (Sequence)'] = pd.to_numeric(df['Alignment length (Sequence)'], errors='coerce')
df['BLOSUM80_per_residue'] = df['BLOSUM80 score'] / df['Alignment length (Sequence)']

df['Alignment_coverage_sequence'] = (df['Alignment length (Sequence)'] / df['pathogen length'] * 100).clip(upper=100)

print("New shape:", df.shape)  # Should now say (399, 25)
print("\nNew columns confirmed:")
print(df[['BLOSUM80_per_residue', 'Alignment_coverage_sequence']].head())


New shape: (399, 25)

New columns confirmed:
   BLOSUM80_per_residue  Alignment_coverage_sequence
0                   NaN                          NaN
1              4.111111                         90.0
2              2.916667                        100.0
3              4.000000                         90.0
4              4.375000                         80.0


In [32]:
columns_to_convert = [
    'BLOSUM80 score',
    'Identity percentage',
    'Alignment length (Sequence)',
    'Identical aa',
    'pathogen length',
    'Human length',
    '%Rank_EL(X)',
    'Aff(nM)(X)',
    'Structural RMSD',
    'TM-align score (Human chain 2)',
    'Structural alignment coverage %',
    'Alignment Length (Structure)'
]

for col in columns_to_convert:
  df[col] = pd.to_numeric(df[col], errors='coerce')

df['BLOSUM80_per_residue'] = df['BLOSUM80 score'] / df['Alignment length (Sequence)']
df['Alignment_coverage_sequence'] = (df['Alignment length (Sequence)'] / df['pathogen length'] * 100).clip(upper=100)
print('All Columns Converted Successfully!')
print('\nMissing values per column:')
print(df.isnull().sum()[df.isnull().sum()>0])

Y_group = df[df['RMSD_Mimic_Target (Y)'] == 'Y']
N_group = df[df['RMSD_Mimic_Target (Y)'] == 'N']

print(f'Y Group_size : {len(Y_group)}')
print(f'N Group_size : {len(N_group)}')

All Columns Converted Successfully!

Missing values per column:
BLOSUM80 score                    127
Identity percentage               127
Alignment length (Sequence)       128
Identical aa                      127
TM-align score (Human chain 2)      1
BLOSUM80_per_residue              128
Alignment_coverage_sequence       128
dtype: int64
Y Group_size : 262
N Group_size : 137


The 127-128 missing values in BLOSUM and sequence alignment columns are the 137 negatives minus the few that happened to have values. These are missing because negatives never went through BLAST. This is not a data quality problem — it is correct behaviour that we already documented. The 1 missing TM-score we knew about from before.